# SKKB — Baseline Judge Run

JIRA: **GCAI-3092** — diagnostic LLM-as-a-Judge evaluation for the SK Knowledge-Base chatbot.

This notebook runs the `hg_ds_evals` judge against the Delta table produced by [`skkb_001_data_preparation.ipynb`](skkb_001_data_preparation.ipynb).

- **YAML config:** [`configs/skkb/skkb_exp_001_baseline.yaml`](../../configs/skkb/skkb_exp_001_baseline.yaml)
- **System template:** [`configs/skkb/system.md.j2`](../../configs/skkb/system.md.j2) — includes the baked-in **AGENT CONTEXT** (main_agent + daily_banking_agent system prompts and tool schemas) so the judge sees what the live agents saw.
- **Judge:** `gpt-5` via `azure_async`, `reasoning_effort: medium`
- **Output:** CSV checkpoint at `checkpoints/evals_skkb_exp_001_baseline_medium.csv`

> **NOTE — why we do not call `run_experiment` directly:** the upstream
> `run_experiment` helper instantiates `PromptBuilder(rubric=rubric)`
> without passing the YAML's `paths.system_template_path`, so the
> default embedded template is used and our `critical_evaluation_rules`,
> `domain_specific_guidance`, `final_reminders`, and the baked-in
> **AGENT CONTEXT** are silently dropped. To keep the custom template
> in effect we replicate `run_experiment`'s flow explicitly below.

For a dry run, set `dataset.test_num_rows: 5` in the YAML before running.


## Environment

In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
# !pip install -qe /Workspace/Repos/Shared_HeyGeorge/hg-ds-evals
# !pip install -qe /Workspace/Repos/Shared_HeyGeorge/hey-george-ds/ds_common
!pip install -qe /Workspace/Users/sg7cb@s-mxs.net/hey-george-ds/ds_common
!pip install openai -q

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
RUN_IN_DBX:bool = True

In [0]:
import config_nbs
config_nbs.add_local_paths()

import hg_ds_evals
print(hg_ds_evals.__file__)

import pandas as pd
pd.set_option("display.width", None)

Path "/Workspace/Users/sg7cb@s-mxs.net/hg-ds-evals/" added!
Path "/Workspace/Users/sg7cb@s-mxs.net/hg-ds-evals/experiments/skkb/" added!
/Workspace/Users/sg7cb@s-mxs.net/hg-ds-evals/hg_ds_evals/__init__.py


## Load the experiment YAML and build the rubric

In [0]:
from pathlib import Path

YAML_FILE_NAME = "skkb_exp_001_baseline"
# config_path = f"/Workspace/Repos/Shared_HeyGeorge/ds-evals/configs/skkb/{YAML_FILE_NAME}.yaml"
config_path = f"../configs/skkb/{YAML_FILE_NAME}.yaml"
assert Path(config_path).exists(), f"config file not found: {config_path}"


In [0]:
from hg_ds_evals.rubrics.loader import (
    load_experiment_config,
    build_rubric_from_config,
    get_experiment_name,
)

config = load_experiment_config(config_path)
rubric = build_rubric_from_config(config)

print(f"Experiment       : {config['experiment']['name']}")
print(f"Rubric name      : {rubric.metadata.name}")
print(f"Rubric persona   : {rubric.metadata.persona}")
print(f"Dimensions       : {rubric.dimension_ids}")
print(f"Input fields     : {rubric.input_field_names}")
print(f"Output fields    : {[f.name for f in rubric.output_schema.fields]}")
print(f"Pass threshold   : {rubric.pass_threshold}")
print(f"Judge model      : {config['model']['model_deployment_name']} "
      f"({config['model']['api_provider']}, reasoning={config['model']['reasoning_effort']})")
print(f"Input table      : {config['dataset']['input_schema']}.{config['dataset']['input_dataset']}")
print(f"test_num_rows    : {config['dataset']['test_num_rows']}")


Experiment       : skkb_exp_001_baseline
Rubric name      : Custom Rubric
Rubric persona   : You are an expert evaluator assessing the end-to-end quality of a Slovak retail-banking knowledge-base (KB) chatbot ('Hey George'), with the goal of pinpointing concrete improvements for the bot/agent team, the retrieval/reranker team, and the KB content team.

Dimensions       : ['query_clarity', 'selection_semantic_relevance', 'selected_context_sufficiency', 'optimal_retrieved_context_adequacy', 'answer_expected_alignment', 'answer_groundedness', 'language_compliance']
Input fields     : ['user_query', 'pre_prune_enum_ids', 'post_prune_candidates_text', 'post_prune_enum_ids', 'reranked_enum_ids', 'reranked_kb_context', 'expected_enums', 'expected_response', 'agent_response']
Output fields    : ['case_scope', 'expected_reference_looks_wrong', 'expected_reference_issue_description', 'optimal_enum_selection', 'expected_answer_summary_with_optimal_context', 'extra_or_distracting_enums', 'unavaila

## Load the input data

In [0]:
input_table = f"{config['dataset']['input_catalog']}.{config['dataset']['input_schema']}.{config['dataset']['input_dataset']}"
print(input_table)

prod_aut_chat00_lab_catalog.private_uc0115_aix_db.dts_eval_skkb_exp_001_online_nightly_2026_04_24_infer


In [0]:
df_input = spark.read.table(input_table)
print(df_input.count())

484


In [0]:
display(df_input)

test_case_id trace_id request_time execution_duration_ms user_query knowledge_search_run_count knowledge_search_final_run_index knowledge_search_runs reranked_kb_context kb_version reranked_enum_ids raw_vector_db_retrieved_candidates_text raw_vector_db_retrieved_enum_ids raw_vector_db_retrieved_enum_count raw_vector_db_query_count raw_vector_db_retrieved_count_by_query raw_vector_db_query_limits pre_prune_candidates_text pre_prune_enum_ids pre_prune_enum_count post_prune_candidates_text post_prune_enum_ids post_prune_enum_count prune_counts_available prune_candidates_in prune_candidates_out prune_candidates_dropped agent_response expected_response expected_enums expected_enums_weights expert_score expert_rationale enum_relevance_score agents_called tools_called query_scope trace_invariant_violations reranker_selected_empty reranker_raw_selected_ids reranker_valid_selected_ids reranker_invalid_selected_ids reranker_unselected_context_ids reranker_selection_status reranker_selection_violations reranker_system_prompt reranker_user_prompt reranker_model reranker_token_usage main_agent_prompt_hash daily_banking_agent_prompt_hash user_query_en categories_list expected_response_en reranked_enums_kb_sk reranked_enums_kb_en post_prune_candidates_kb_sk post_prune_candidates_kb_en agent_response_en execution_duration_s execution_duration_min Test case 512 tr-3f31a1fc540484398cfab33b318fa7f9 1777036340022 13618 Potrebujem niečo riešiť ohľadom účtu, môžem napísať priamo svojmu poradcovi? 1 1 [{"run_index": 1, "knowledge_search_span_id": "CAy1JRuQFO0=", "knowledge_search_start_time": 1777036344270050000, "knowledge_search_end_time": 1777036347093350000, "reranked_kb_context": "[KB_GAI_SK_SK_2026-04-20_14h16_phase_1_2]\nWRITE_MESSAGE: **Informácie o tom, ako získať kontakt na svojho konzultanta/poradcu:**\\nMeno vášho konzultanta aj adresu jeho pobočky nájdete v aplikácii George. Stačí na hlavnej stránke v prehľade produktov prejsť na spodnú časť obrazovky a kliknúť na možnosť Kontakty.\\nZobrazí sa vám meno konzultanta spolu s adresou pobočky na ktorej pôsobí.\\nAk ho chcete kontaktovať, môžete mu priamo z Georgea odoslať správu cez možnosť Poslať správu. \\nRovnako si s ním môžete dohodnúť stretnutie prostredníctvom možnosti Dohodnúť stretnutie.\\n**Informácie o tom, ako napísať konzultantovi/poradcovi správu:**\\nAk chcete napísať správu svojmu konzultantovi, stačí na hlavnej stránke v prehľade produktov prejsť na spodnú časť obrazovky a kliknúť na možnosť Kontakty. Otvorí sa vám stránka s informáciami o vašom konzultantovi, kde nájdete aj jeho meno a tlačidlo Poslať správu.\\nNásledne si zo zoznamu dostupných tém vyberiete oblasť, o ktorej chcete písať, napíšete svoju otázku, prípadne priložíte súbor -- a hotovo, správu môžete odoslať.\\nAk máte v Georgeovi povolené upozornenia, o odpovedi od konzultanta sa dozviete hneď, ako vám ju pošle.\\n**Informácie o tom, ako odpovedať na správu doručenú od konzultanta/poradcu: **\\nAk chcete odpovedať na správu, ktorá vám prišla od konzultanta, stačí na hlavnej stránke v prehľade produktov prejsť na spodnú časť obrazovky a kliknúť na možnosť Kontakty. V sekcii Doručené správny vyberte možnosť Messenger. Tu sa zobrazia vaše doručené správy, kde kliknutím na konkrétnu správu môžete zaslať odpoveď. \\n\nCALL_BRANCH_AUTHORISED: **Ako zatelefonovať poradcovi do banky cez Georgea**\\nV súčasnosti nie je možné volať pracovníkovi pobočky priamo cez aplikáciu George. \\nPracovníkovi pobočky však môžete jednoducho poslať písomnú správu cez George Messenger.\\n**Telefonické kontaktovanie Klientskeho centra cez Georgea**\\nV prípade potreby je možné Klientske centrum jednoducho a rýchlo kontaktovať prostredníctvom aplikácie George.\\nV mobilnej aplikácii George kliknite vpravo dole na Kontakty. V časti Klientske centrum zvoľte možnosť Zavolať.\\nNásledne je potrebné vybrať si jednu z možností volania:\\n- Klientske centrum\\n- Klientske centrum zo zahraničia\\nIde o bežný telefónny, nie o internetový hovor, a

## Build the judge's system + user prompts from the **custom** template

We explicitly pass `system_template_path` and `user_template_path` — without this the default embedded template is used and the AGENT CONTEXT / critical_evaluation_rules / final_reminders sections are silently dropped.

In [0]:
from hg_ds_evals.prompts.builder import PromptBuilder
from hg_ds_evals.prompts.common import display_prompt

system_template_path = Path(config["paths"]["system_template_path"])
user_template_path = Path(config["paths"]["user_template_path"])

# If the YAML paths are relative to the notebook, resolve them
if not system_template_path.is_absolute():
    system_template_path = (Path.cwd() / system_template_path).resolve()
if not user_template_path.is_absolute():
    user_template_path = (Path.cwd() / user_template_path).resolve()

assert system_template_path.exists(), f"system template missing: {system_template_path}"
assert user_template_path.exists(), f"user template missing: {user_template_path}"
print(f"system template : {system_template_path}")
print(f"user template   : {user_template_path}")

builder = PromptBuilder(
    rubric=rubric,
    system_template_path=system_template_path,
    user_template_path=user_template_path,
)
system_prompt = builder.build_system_prompt()
print(f"system prompt   : {len(system_prompt)} chars")


system template : /Workspace/Users/sg7cb@s-mxs.net/hg-ds-evals/experiments/skkb/configs/skkb/system.md.j2
user template   : /Workspace/Users/sg7cb@s-mxs.net/hg-ds-evals/experiments/skkb/configs/skkb/user.md.j2
system prompt   : 25355 chars


In [0]:
display_prompt(system_prompt, title="System Prompt (SKKB baseline — with AGENT CONTEXT)", font_size=12)

## Preview the user prompt on one real row

In [0]:
sample_row = (
    spark.read.table(input_table)
    .limit(1)
    .toPandas()
    .iloc[0]
    .to_dict()
)

user_prompt = builder.build_user_prompt(sample_row)
display_prompt(user_prompt, title=f"User Prompt — {sample_row.get('test_case_id')}", font_size=12)


## Run the evaluation

Steps: load the Spark table, prepare the eval sample, filter by the checkpoint, set up the Azure client, and call `async_run_evals` with the **custom** system_prompt and user_prompt_builder.


In [0]:
from pyspark.sql import SparkSession
from hg_ds_evals.common.utils import (
    load_checkpoint,
    filter_df_with_checkpoints,
    prepare_eval_sample,
    )
from hg_ds_evals.llm.api_client import get_api_client
from hg_ds_evals.evals.evaluator import async_run_evals

spark = SparkSession.builder.getOrCreate()

experiment_name = get_experiment_name(config_path)
INPUT_CATALOG = config["dataset"]["input_catalog"]
INPUT_SCHEMA = config["dataset"]["input_schema"]
INPUT_TABLE = config["dataset"]["input_dataset"]
ID_COLUMNS = config["dataset"].get("id_columns", [])
NUM_ROWS = config["dataset"]["test_num_rows"]
CHECKPOINT_DIR = config["paths"].get("checkpoint_dir", "checkpoints")

print(f"[1/5] loading input table {INPUT_CATALOG}.{INPUT_SCHEMA}.{INPUT_TABLE}")
df = spark.read.table(f"{INPUT_CATALOG}.{INPUT_SCHEMA}.{INPUT_TABLE}")
print(f"      rows: {df.count()}")

print("\n[2/5] preparing eval sample")
df_sample, file_name_eval = prepare_eval_sample(
    df=df,
    evals_name=experiment_name,
    reasoning_effort=config["model"]["reasoning_effort"],
    num_rows=NUM_ROWS,
)
cols = config["dataset"]["eval_columns"]
if ID_COLUMNS:
    cols = list(dict.fromkeys(ID_COLUMNS + cols))
passthrough = config["dataset"].get("passthrough_columns", [])
if passthrough:
    cols = list(dict.fromkeys(cols + passthrough))
df_eval = df_sample[cols].copy()
print(f"      eval df: {len(df_eval)} rows, {len(cols)} cols  checkpoint_file={file_name_eval}")

print("\n[3/5] loading checkpoint")
ckp_df, ckp_path = load_checkpoint(
    checkpoint_file_name=file_name_eval,
    checkpoint_dir=CHECKPOINT_DIR,
)
df_eval = filter_df_with_checkpoints(df_eval, ckp_df, id_cols=ID_COLUMNS)
print(f"      {len(df_eval)} rows remain after filtering {len(ckp_df)} already-scored rows")

print("\n[4/5] setting up API client")
client = get_api_client(
    model_deployment_name=config["model"]["model_deployment_name"],
    api_provider=config["model"]["api_provider"],
    databricks_endpoint_url=config["model"].get("databricks_endpoint_url"),
    databricks_base_url=config["model"].get("databricks_base_url"),
    databricks_workspace_host=config["model"].get("databricks_workspace_host"),
)

print("\n[5/5] running evaluations")
results_df, metrics = await async_run_evals(
    df=df_eval,
    system_prompt=system_prompt,
    client=client,
    config=config,
    checkpoint_path=ckp_path,
    user_prompt_builder=builder.build_user_prompt,
)

print("\ndone.")
print(f"checkpoint  : {ckp_path}")
print(f"metrics     : {metrics}")


[1/5] loading input table prod_aut_chat00_lab_catalog.private_uc0115_aix_db.dts_eval_skkb_exp_001_online_nightly_2026_04_24_infer
      rows: 484

[2/5] preparing eval sample
Sample size: 5
File name: evals_skkb_exp_001_baseline_high_test.csv
      eval df: 5 rows, 51 cols  checkpoint_file=evals_skkb_exp_001_baseline_high_test.csv

[3/5] loading checkpoint
ℹ No checkpoint found, starting fresh: checkpoints/databricks_gpt51/evals_skkb_exp_001_baseline_high_test.csv
      5 rows remain after filtering 0 already-scored rows

[4/5] setting up API client

[5/5] running evaluations
ℹ️ Processing batch 1: 5 tasks
✅ Batch 1 complete: 131,524 tokens in 58.0s | Last 60s: 131,524 tokens
✅ All batches complete: 5 rows, 131,524 tokens, 58.0s, avg TPM: 136,039

done.
checkpoint  : checkpoints/databricks_gpt51/evals_skkb_exp_001_baseline_high_test.csv
metrics     : {0: {'time_elapsed_sec': 56.589664697647095, 'input_tokens': 20729, 'output_tokens': 4370, 'total_tokens': 25099, 'parse_retries': 0}, 1:

[Trace(trace_id=tr-76187cec2abd2ad9915b335795fb4763), Trace(trace_id=tr-00fc2ddbdad1cced9f4632f0c9231621), Trace(trace_id=tr-32992f1294cec77fc2a9e1e7d7e15661), Trace(trace_id=tr-7691bbe41739a3d08d04700ccd107687), Trace(trace_id=tr-5f1b583e8f16f097e6e8fc15049062bc)]

## Smoke check the checkpoint

In [0]:
import pandas as pd

df_results = pd.read_csv(ckp_path)
print(f"checkpoint rows: {len(df_results)}")
# print(f"columns        : {list(df_results.columns)}")
print()
print("error counts:")
print(df_results.get("error", pd.Series([None]*len(df_results))).isna().value_counts(dropna=False))

dim_score_cols = [c for c in df_results.columns if c.endswith("_score")]
print()
print("dimension score summary:")
print(df_results[dim_score_cols].describe().round(2))


checkpoint rows: 5

error counts:
error
False    5
Name: count, dtype: int64

dimension score summary:
       expert_score  ...  language_compliance_score
count          5.00  ...                        5.0
mean           6.60  ...                        2.0
std            1.34  ...                        0.0
min            6.00  ...                        2.0
25%            6.00  ...                        2.0
50%            6.00  ...                        2.0
75%            6.00  ...                        2.0
max            9.00  ...                        2.0

[8 rows x 9 columns]


In [0]:
cols = ["test_case_id", 
        "user_query", "user_query_en",
        "reranked_enum_ids", "retrieved_enum_ids", "expected_enums",
        'reranker_selected_ids', 'reranker_raw_selected_ids', 'reranker_valid_selected_ids', 'reranker_invalid_selected_ids', 'reranker_unselected_context_ids', 'reranker_selection_status', 'reranker_selection_violations']
display(df_results[cols].sort_values(by="test_case_id", ascending=True).head(25))

---------------------------------------------------------------------------
KeyError                                  Traceback (most recent call last)
File <command-7071891307303841>, line 5
      1 cols = ["test_case_id", 
      2         "user_query", "user_query_en",
      3         "reranked_enum_ids", "retrieved_enum_ids", "expected_enums",
      4         'reranker_selected_ids', 'reranker_raw_selected_ids', 'reranker_valid_selected_ids', 'reranker_invalid_selected_ids', 'reranker_unselected_context_ids', 'reranker_selection_status', 'reranker_selection_violations']
----> 5 display(df_results[cols].sort_values(by="test_case_id", ascending=True).head(25))

File /databricks/python/lib/python3.12/site-packages/pandas/core/frame.py:4108, in DataFrame.__getitem__(self, key)
   4106     if is_iterator(key):
   4107         key = list(key)
-> 4108     indexer = self.columns._get_indexer_strict(key, "columns")[1]
   4110 # take() does not accept boolean indexers
   4111 if getattr(index

## Next step

Open [`skkb_001_results_viewer.ipynb`](skkb_001_results_viewer.ipynb) to aggregate scores, compute the judge-vs-expert Spearman, split missed ENUMs into retriever-vs-reranker failures, and render the HTML report.
